# Silver PSGC Population
The Silver layer transforms the nested PSGC population payload from `public_health.bronze.geography_bronze` into clean, typed municipal population observations.

Silver grain: one row per psgc_code × population_year × source_version.


1. Parse the nested population JSON

In [ ]:
%sql

CREATE OR REPLACE TEMP VIEW psgc_population_observations AS

SELECT
    g.psgc_code AS parent_psgc_code,
    g.area_name AS parent_area_name,
    g.geographic_level AS parent_geographic_level,
    g.source_version,
    g.batch_id,

    population_item.psgc AS population_psgc_code,
    population_item.year AS population_year,
    population_item.population AS population_raw,

    TRY_CAST(
        REGEXP_REPLACE(
            TRIM(population_item.population),
            ',',
            ''
        ) AS BIGINT
    ) AS population

FROM public_health.bronze.geography_bronze g

LATERAL VIEW EXPLODE(
    FROM_JSON(
        g.populations_json,
        'array<struct<id:bigint,population:string,psgc:string,year:int>>'
    )
) exploded AS population_item;


2. Classify population parsing results

    The raw population value is trimmed and converted to a numeric BIGINT.
    - MISSING: no population value was supplied
    - INVALID: a non-null population value cannot be parsed as a number
    - VALID: population was successfully parsed

In [ ]:
%sql

CREATE OR REPLACE TEMP VIEW psgc_population_parsed AS

SELECT
    parent_psgc_code,
    parent_area_name,
    parent_geographic_level,
    source_version,
    batch_id,
    population_psgc_code,
    population_year,
    population_raw,
    population,

    CASE
        WHEN population_raw IS NULL
            THEN 'MISSING'
        WHEN population IS NULL
            THEN 'INVALID'
        ELSE 'VALID'
    END AS population_parse_status

FROM psgc_population_observations;


3. Profile the population observations

    This is an inspection step. It does not modify the data.

    The Bronze QC identified 94 non-null population values that could not be parsed as BIGINT; these records are expected to be excluded from the clean Silver population table.

In [ ]:
%sql

SELECT
    source_version,
    parent_geographic_level,
    COUNT(*) AS total_observations,
    COUNT_IF(population_parse_status = 'VALID') AS valid_population,
    COUNT_IF(population_parse_status = 'MISSING') AS missing_population,
    COUNT_IF(population_parse_status = 'INVALID') AS invalid_population,
    COUNT(DISTINCT population_psgc_code) AS distinct_population_psgc_codes
FROM psgc_population_parsed
GROUP BY
    source_version,
    parent_geographic_level
ORDER BY
    source_version,
    parent_geographic_level;


4. Validate PSGC identity consistency

    Each population observation should refer to the same PSGC geography as the Bronze record that contains it.

    Records with a missing population PSGC or a parent/population PSGC mismatch are excluded from Silver.

In [ ]:
%sql

SELECT
    parent_geographic_level,
    COUNT(*) AS total_observations,
    COUNT_IF(population_psgc_code IS NULL) AS missing_population_code,
    COUNT_IF(
        population_psgc_code IS NOT NULL
        AND parent_psgc_code <> population_psgc_code
    ) AS parent_population_mismatch
FROM psgc_population_parsed
GROUP BY parent_geographic_level
ORDER BY parent_geographic_level;


5. Create the municipal PSGC population Silver table

    Cleaning and transformation:
    1. Keep only observations with a population PSGC code.
    2. Keep only observations whose population value parsed successfully.
    3. Keep only non-negative population values.
    4. Keep only observations with a non-null, valid four-digit population year.
    5. Require the population PSGC to match the parent PSGC from the Bronze record.
    6. Join to `geography_silver` so that only PSGC codes mapped to a municipality (Mun) are retained.
    7. Remove raw JSON parsing fields from the analytical Silver output; the original values remain available in Bronze.
    8. Preserve source version and batch ID for lineage.

In [ ]:
%sql

CREATE OR REPLACE TABLE public_health.silver.psgc_population_silver
USING DELTA
COMMENT 'Cleaned municipal population observations from PSGC source data'
AS

SELECT
    g.psgc_code,
    p.population_year,
    p.population,
    p.source_version,
    p.batch_id,
    CURRENT_TIMESTAMP() AS silver_processed_at

FROM psgc_population_parsed p

INNER JOIN public_health.silver.geography_silver g
    ON p.population_psgc_code = g.psgc_code
   AND g.geographic_level = 'Mun'

WHERE p.population_psgc_code IS NOT NULL
  AND p.parent_psgc_code = p.population_psgc_code
  AND p.population_year BETWEEN 1900 AND YEAR(CURRENT_DATE())
  AND p.population_parse_status = 'VALID'
  AND p.population IS NOT NULL
  AND p.population >= 0;


6. Check the Silver grain

    The intended Silver grain is:

    `psgc_code × population_year × source_version`

    There should be at most one row at this grain.

In [ ]:
%sql

SELECT
    psgc_code,
    population_year,
    source_version,
    COUNT(*) AS row_count
FROM public_health.silver.psgc_population_silver
GROUP BY
    psgc_code,
    population_year,
    source_version
HAVING COUNT(*) > 1
ORDER BY
    row_count DESC,
    psgc_code,
    population_year;


7. Validate cleaned population values

In [ ]:
%sql

SELECT
    COUNT(*) AS total_rows,
    COUNT_IF(psgc_code IS NULL) AS null_psgc_code,
    COUNT_IF(population_year IS NULL) AS null_population_year,
    COUNT_IF(population IS NULL) AS null_population,
    COUNT_IF(population < 0) AS negative_population,
    MIN(population) AS min_population,
    MAX(population) AS max_population
FROM public_health.silver.psgc_population_silver;


8. Validate municipality mapping

    Every Silver population observation must map to a municipality in `geography_silver`.

    This should return no rows.

In [ ]:
%sql

SELECT
    p.population_psgc_code,
    COUNT(*) AS observation_count
FROM psgc_population_parsed p

LEFT JOIN public_health.silver.geography_silver g
    ON p.population_psgc_code = g.psgc_code
   AND g.geographic_level = 'Mun'

WHERE p.population_psgc_code IS NOT NULL
  AND p.population_parse_status = 'VALID'
  AND g.psgc_code IS NULL

GROUP BY p.population_psgc_code
ORDER BY p.population_psgc_code;


9. Validate parent/population PSGC consistency in the Silver input

    This is an additional diagnostic to make mismatches visible before they are filtered from the Silver table.

    The query should return no rows for observations eligible for Silver.

In [ ]:
%sql

SELECT
    parent_psgc_code,
    population_psgc_code,
    population_year,
    COUNT(*) AS observation_count
FROM psgc_population_parsed
WHERE population_parse_status = 'VALID'
  AND population_psgc_code IS NOT NULL
  AND parent_psgc_code <> population_psgc_code
GROUP BY
    parent_psgc_code,
    population_psgc_code,
    population_year
ORDER BY observation_count DESC;
